# PPE Detection + Pose Estimation 앙상블 프로토타입

**Detection 모델**: 직접 학습한 YOLOv8n (person / helmet / vest)

**Pose 모델**: YOLOv8n-pose (COCO pretrained)

**판단 로직**: Detection bbox + Pose keypoint → 착용 / 휴대 / 미착용

## 1. 환경 설정

In [ ]:
!nvidia-smi
!pip install ultralytics==8.3.40 -q

Mon May 11 11:20:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
import cv2
import numpy as np
import torch
import time
import io
import PIL.Image
from ultralytics import YOLO
from IPython.display import display, Javascript, clear_output, Image as IPImage

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


## 2. 모델 로드

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os # os 모듈을 임포트하여 파일 존재 여부를 확인합니다.

# ============================================
# 경로 설정 (본인 환경에 맞게 수정)
# ============================================
DET_MODEL_PATH = '/content/drive/MyDrive/PPE_Model/best_260409.pt'

# 파일 존재 여부 확인
if not os.path.exists(DET_MODEL_PATH):
    raise FileNotFoundError(
        f"'{DET_MODEL_PATH}' 파일을 찾을 수 없습니다. "
        "Google Drive에 모델 파일이 올바르게 업로드되었는지 확인하거나, "
        "DET_MODEL_PATH 경로를 수정해 주세요."
    )

# 모델 로드
det_model = YOLO(DET_MODEL_PATH)
pose_model = YOLO('yolov8n-pose.pt')

# 클래스 확인
print(f"\nDetection 클래스: {det_model.names}")
print(f"Pose 모델: yolov8n-pose (COCO 17 keypoints)")

# 클래스 인덱스 자동 매핑
CLASS_MAP = {}  # idx -> 'person'/'helmet'/'vest'
for idx, name in det_model.names.items():
    name_lower = name.lower()
    if 'person' in name_lower:
        CLASS_MAP[idx] = 'person'
    elif 'helmet' in name_lower or 'hardhat' in name_lower:
        CLASS_MAP[idx] = 'helmet'
    elif 'vest' in name_lower:
        CLASS_MAP[idx] = 'vest'

print(f"\n사용할 클래스 매핑: {CLASS_MAP}")
assert 'person' in CLASS_MAP.values(), "person 클래스를 찾을 수 없습니다!"
assert 'helmet' in CLASS_MAP.values(), "helmet 클래스를 찾을 수 없습니다!"
assert 'vest' in CLASS_MAP.values(), "vest 클래스를 찾을 수 없습니다!"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: '/content/drive/MyDrive/PPE_Model/best_260409.pt' 파일을 찾을 수 없습니다. Google Drive에 모델 파일이 올바르게 업로드되었는지 확인하거나, DET_MODEL_PATH 경로를 수정해 주세요.

## 3. 앙상블 판단 알고리즘

In [ ]:
# ============================================
# COCO Pose Keypoint 인덱스
# ============================================
# 0:코  1:왼눈  2:오른눈  3:왼귀  4:오른귀
# 5:왼어깨  6:오른어깨  7:왼팔꿈치  8:오른팔꿈치
# 9:왼손목  10:오른손목  11:왼엉덩이  12:오른엉덩이
# 13:왼무릎  14:오른무릎  15:왼발목  16:오른발목

HEAD_KP = [0, 1, 2, 3, 4]
SHOULDER_KP = [5, 6]
KP_CONF_THRESH = 0.3


def get_valid_center(keypoints, kp_conf, indices, thresh=KP_CONF_THRESH):
    """
    지정된 keypoint 인덱스에서 confidence가 충분한 것들의 중심 좌표.
    Returns: (x, y) or None
    """
    pts = []
    for i in indices:
        if kp_conf[i] > thresh and keypoints[i][0] > 0 and keypoints[i][1] > 0:
            pts.append(keypoints[i])
    if not pts:
        return None
    pts = np.array(pts)
    return (float(np.mean(pts[:, 0])), float(np.mean(pts[:, 1])))


def calc_iou(a, b):
    """두 bbox (x1,y1,x2,y2)의 IoU."""
    ix1 = max(a[0], b[0])
    iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2])
    iy2 = min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1])) - inter
    return inter / union if union > 0 else 0


def judge_helmet(person_box, helmet_boxes, head_center=None):
    """
    헬멧 착용 상태 판단.

    로직:
      1. person bbox 안에 helmet bbox가 있는지
      2. 있으면 → 머리 keypoint 근처인지 (Pose ON)
                   또는 person 상단 30% 이내인지 (Pose OFF)
      3. 머리 근처 → 'worn' / 아래쪽 → 'carried' / 없으면 → 'not_detected'
    """
    px1, py1, px2, py2 = person_box
    p_h = py2 - py1
    p_w = px2 - px1

    # person 안에 있는 helmet 필터링
    inside = []
    for h in helmet_boxes:
        hcx = (h['bbox'][0] + h['bbox'][2]) / 2
        hcy = (h['bbox'][1] + h['bbox'][3]) / 2
        # helmet 중심이 person bbox 내부 (10% 마진)
        if (px1 - p_w*0.1 <= hcx <= px2 + p_w*0.1 and
            py1 - p_h*0.1 <= hcy <= py2 + p_h*0.1):
            inside.append(h)

    if not inside:
        return 'not_detected'

    for h in inside:
        hcx = (h['bbox'][0] + h['bbox'][2]) / 2
        hcy = (h['bbox'][1] + h['bbox'][3]) / 2

        if head_center is not None:
            # Pose 기반 판단
            hdx, hdy = head_center
            dy = hdy - hcy   # 양수 → helmet이 머리보다 위
            dx = abs(hdx - hcx)

            # 헬멧이 머리 위~약간 아래, 수평으로 가까우면 → 착용
            if dy > -p_h * 0.15 and dx < p_w * 0.4:
                return 'worn'
        else:
            # Fallback: person bbox 상단 30% 이내이면 착용
            if hcy < py1 + p_h * 0.30:
                return 'worn'

    # helmet이 있긴 한데 머리 근처가 아님 → 휴대
    return 'carried'


def judge_vest(person_box, vest_boxes, shoulder_center=None):
    """
    조끼 착용 상태 판단.

    로직:
      1. person bbox 안에 vest bbox가 있는지
      2. 있으면 → 어깨 keypoint 근처 상체 영역인지 (Pose ON)
                   또는 person 중간 영역(15%~70%)인지 (Pose OFF)
      3. 상체 영역 → 'worn' / 없으면 → 'not_detected'
    """
    px1, py1, px2, py2 = person_box
    p_h = py2 - py1
    p_w = px2 - px1

    inside = []
    for v in vest_boxes:
        vcx = (v['bbox'][0] + v['bbox'][2]) / 2
        vcy = (v['bbox'][1] + v['bbox'][3]) / 2
        if (px1 - p_w*0.1 <= vcx <= px2 + p_w*0.1 and
            py1 - p_h*0.1 <= vcy <= py2 + p_h*0.1):
            inside.append(v)

    if not inside:
        return 'not_detected'

    for v in inside:
        vcy = (v['bbox'][1] + v['bbox'][3]) / 2

        if shoulder_center is not None:
            sy = shoulder_center[1]
            # vest가 어깨 부근 ~ 허리 사이 → 착용
            if abs(vcy - sy) < p_h * 0.35:
                return 'worn'
        else:
            rel_y = (vcy - py1) / p_h
            if 0.15 <= rel_y <= 0.70:
                return 'worn'

    return 'not_detected'


print("판단 알고리즘 로드 완료")

## 4. 통합 추론 파이프라인

In [ ]:
def run_pipeline(frame, det_model, pose_model,
                 det_conf=0.4, use_pose=True):
    """
    전체 파이프라인: Detection → Pose → 매칭 → 판단

    Returns:
        persons: [{
            'bbox': [x1,y1,x2,y2],
            'conf': float,
            'helmet': 'worn'/'carried'/'not_detected',
            'vest':   'worn'/'not_detected',
            'head_center': (x,y) or None,
            'shoulder_center': (x,y) or None,
            'keypoints': ndarray(17,2) or None,
            'kp_conf': ndarray(17,) or None,
            'safe': bool
        }]
    """
    # ---- Step 1: Detection ----
    det = det_model.predict(frame, imgsz=640, conf=det_conf,
                            iou=0.5, verbose=False)[0]

    person_list = []
    helmet_list = []
    vest_list = []

    if det.boxes is not None:
        for box in det.boxes:
            cid = int(box.cls[0])
            if cid not in CLASS_MAP:
                continue
            label = CLASS_MAP[cid]
            item = {
                'bbox': box.xyxy[0].cpu().numpy().tolist(),
                'conf': float(box.conf[0])
            }
            if label == 'person':
                person_list.append(item)
            elif label == 'helmet':
                helmet_list.append(item)
            elif label == 'vest':
                vest_list.append(item)

    # ---- Step 2: Pose (선택적) ----
    pose_kps_list = []   # [(bbox, keypoints(17,2), kp_conf(17,))]
    if use_pose:
        pose = pose_model.predict(frame, imgsz=640, conf=0.4, verbose=False)[0]
        if pose.keypoints is not None and pose.boxes is not None:
            for i in range(len(pose.boxes)):
                pb = pose.boxes[i].xyxy[0].cpu().numpy().tolist()
                kp = pose.keypoints[i].data[0].cpu().numpy()  # (17, 3)
                pose_kps_list.append((pb, kp[:, :2], kp[:, 2]))

    # ---- Step 3: Person ↔ Pose 매칭 (IoU 기반) ----
    results = []
    for p in person_list:
        head_c = None
        shoulder_c = None
        matched_kps = None
        matched_conf = None

        if pose_kps_list:
            best_iou = 0
            best_idx = -1
            for k_idx, (pb, kps, kc) in enumerate(pose_kps_list):
                iou = calc_iou(p['bbox'], pb)
                if iou > best_iou:
                    best_iou = iou
                    best_idx = k_idx

            if best_iou > 0.5 and best_idx >= 0:
                _, kps, kc = pose_kps_list[best_idx]
                head_c = get_valid_center(kps, kc, HEAD_KP)
                shoulder_c = get_valid_center(kps, kc, SHOULDER_KP)
                matched_kps = kps
                matched_conf = kc

        # ---- Step 4: 착용 판단 ----
        h_status = judge_helmet(p['bbox'], helmet_list, head_c)
        v_status = judge_vest(p['bbox'], vest_list, shoulder_c)

        is_safe = (h_status == 'worn' and v_status == 'worn')

        results.append({
            'bbox': p['bbox'],
            'conf': p['conf'],
            'helmet': h_status,
            'vest': v_status,
            'head_center': head_c,
            'shoulder_center': shoulder_c,
            'keypoints': matched_kps,
            'kp_conf': matched_conf,
            'safe': is_safe
        })

    return results


print("파이프라인 로드 완료")

## 5. 시각화

In [ ]:
# 색상 정의 (BGR)
C_SAFE    = (0, 200, 0)     # 초록 - 안전
C_WARN    = (0, 180, 255)   # 주황 - 휴대
C_DANGER  = (0, 0, 220)     # 빨강 - 미착용
C_KP_HEAD = (255, 0, 255)   # 분홍 - 머리 keypoint
C_KP_SHLD = (255, 255, 0)   # 시안 - 어깨 keypoint
C_CENTER  = (0, 255, 255)   # 노랑 - 중심점 마커

STATUS_COLOR = {
    'worn': C_SAFE,
    'carried': C_WARN,
    'not_detected': C_DANGER,
}


def draw_frame(frame, persons, show_kp=True, show_legend=True):
    """결과를 프레임에 그리기."""
    out = frame.copy()

    violations = 0

    for i, p in enumerate(persons):
        x1, y1, x2, y2 = [int(v) for v in p['bbox']]

        # person bbox 색상
        if p['safe']:
            color = C_SAFE
        elif p['helmet'] == 'carried':
            color = C_WARN
        else:
            color = C_DANGER
            violations += 1

        # person bbox
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)

        # 상태 라벨 배경
        label_h = f"H:{p['helmet']}"
        label_v = f"V:{p['vest']}"
        pose_tag = "P" if p['head_center'] is not None else "-"

        bg_h = 50
        cv2.rectangle(out, (x1, y1 - bg_h), (x1 + 200, y1), (30, 30, 30), -1)

        cv2.putText(out, label_h, (x1 + 4, y1 - 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, STATUS_COLOR[p['helmet']], 1)
        cv2.putText(out, label_v, (x1 + 4, y1 - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, STATUS_COLOR[p['vest']], 1)
        cv2.putText(out, f"[{pose_tag}]", (x1 + 160, y1 - 20),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (180, 180, 180), 1)

        # Keypoint 시각화
        if show_kp and p['keypoints'] is not None and p['kp_conf'] is not None:
            kps = p['keypoints']
            kc = p['kp_conf']

            for idx in HEAD_KP:
                if kc[idx] > KP_CONF_THRESH:
                    cx, cy = int(kps[idx][0]), int(kps[idx][1])
                    cv2.circle(out, (cx, cy), 4, C_KP_HEAD, -1)

            for idx in SHOULDER_KP:
                if kc[idx] > KP_CONF_THRESH:
                    cx, cy = int(kps[idx][0]), int(kps[idx][1])
                    cv2.circle(out, (cx, cy), 4, C_KP_SHLD, -1)

        # 머리 중심 마커
        if p['head_center'] is not None:
            hx, hy = int(p['head_center'][0]), int(p['head_center'][1])
            cv2.drawMarker(out, (hx, hy), C_CENTER,
                          cv2.MARKER_CROSS, 12, 2)

    # 범례
    if show_legend:
        h = out.shape[0]
        cv2.rectangle(out, (0, h-75), (220, h), (30, 30, 30), -1)
        cv2.putText(out, "worn", (10, h-55), cv2.FONT_HERSHEY_SIMPLEX, 0.45, C_SAFE, 1)
        cv2.putText(out, "carried", (10, h-35), cv2.FONT_HERSHEY_SIMPLEX, 0.45, C_WARN, 1)
        cv2.putText(out, "not_detected", (10, h-15), cv2.FONT_HERSHEY_SIMPLEX, 0.45, C_DANGER, 1)
        cv2.circle(out, (140, h-58), 4, C_KP_HEAD, -1)
        cv2.putText(out, "head kp", (150, h-53), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (200,200,200), 1)
        cv2.circle(out, (140, h-38), 4, C_KP_SHLD, -1)
        cv2.putText(out, "shoulder kp", (150, h-33), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (200,200,200), 1)

    return out, violations


print("시각화 함수 로드 완료")

## 6. 테스트 이미지로 검증

In [ ]:
# 테스트 이미지 준비
# 방법 1: 직접 업로드
from google.colab import files as colab_files

print("테스트 이미지를 업로드하세요 (jpg/png)")
print("(건너뛰려면 아래 셀의 샘플 이미지 사용)")
try:
    uploaded = colab_files.upload()
    os.makedirs('/content/test_images', exist_ok=True)
    for fname, data in uploaded.items():
        with open(f'/content/test_images/{fname}', 'wb') as f:
            f.write(data)
    print(f"{len(uploaded)}개 파일 업로드 완료")
except Exception as e:
    print(f"업로드 건너뜀: {e}")

In [ ]:
# 방법 2: 샘플 이미지 다운로드 (위에서 업로드 안 했으면)
os.makedirs('/content/test_images', exist_ok=True)

if len(os.listdir('/content/test_images')) == 0:
    !wget -q -O /content/test_images/sample1.jpg \
        "https://github.com/snehilsanyal/Construction-Site-Safety-PPE-Detection/raw/main/source_files/construction-safety.jpg"
    !wget -q -O /content/test_images/sample2.jpg \
        "https://github.com/snehilsanyal/Construction-Site-Safety-PPE-Detection/raw/main/source_files/two-young-construction-workers-wearing-555864.jpg"
    print("샘플 이미지 다운로드 완료")
else:
    print(f"이미 {len(os.listdir('/content/test_images'))}개 이미지 있음")

In [ ]:
# 이미지별 추론 + 결과
for img_name in sorted(os.listdir('/content/test_images')):
    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    img_path = f'/content/test_images/{img_name}'
    frame = cv2.imread(img_path)
    if frame is None:
        continue

    print(f"\n{'='*60}")
    print(f" {img_name} ({frame.shape[1]}x{frame.shape[0]})")
    print(f"{'='*60}")

    # Pose ON 추론
    t0 = time.time()
    persons = run_pipeline(frame, det_model, pose_model, use_pose=True)
    dt = time.time() - t0

    print(f" 추론 시간: {dt*1000:.0f}ms | 감지 인원: {len(persons)}명")
    print(f" {'─'*56}")

    for j, p in enumerate(persons):
        pose_yn = 'O' if p['head_center'] else 'X'
        safe_yn = 'SAFE' if p['safe'] else 'VIOLATION'
        print(f"  #{j+1} | Helmet: {p['helmet']:14s} | "
              f"Vest: {p['vest']:14s} | Pose: {pose_yn} | {safe_yn}")

    # 시각화
    vis, v_count = draw_frame(frame, persons)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    pil = PIL.Image.fromarray(vis_rgb)
    if pil.width > 800:
        r = 800 / pil.width
        pil = pil.resize((800, int(pil.height * r)))
    display(pil)

    if v_count > 0:
        print(f"  ⚠️  위반 {v_count}건 감지")

## 7. Pose ON vs OFF 비교

In [ ]:
# A/B 비교 (발표 자료용으로 유용)

for img_name in sorted(os.listdir('/content/test_images')):
    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    frame = cv2.imread(f'/content/test_images/{img_name}')
    if frame is None:
        continue

    print(f"\n{'='*60}")
    print(f" {img_name} — Pose ON vs OFF")

    res_off = run_pipeline(frame, det_model, pose_model, use_pose=False)
    res_on  = run_pipeline(frame, det_model, pose_model, use_pose=True)

    vis_off, _ = draw_frame(frame, res_off, show_kp=False, show_legend=False)
    vis_on,  _ = draw_frame(frame, res_on,  show_kp=True,  show_legend=False)

    cv2.putText(vis_off, "Pose OFF (bbox only)", (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    cv2.putText(vis_on,  "Pose ON (keypoint)", (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

    # 높이 맞추고 좌우 병합
    h = max(vis_off.shape[0], vis_on.shape[0])
    if vis_off.shape[0] < h:
        vis_off = cv2.copyMakeBorder(vis_off, 0, h-vis_off.shape[0], 0, 0, cv2.BORDER_CONSTANT)
    if vis_on.shape[0] < h:
        vis_on = cv2.copyMakeBorder(vis_on, 0, h-vis_on.shape[0], 0, 0, cv2.BORDER_CONSTANT)

    combined = np.hstack([vis_off, vis_on])
    pil = PIL.Image.fromarray(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
    if pil.width > 1200:
        r = 1200 / pil.width
        pil = pil.resize((1200, int(pil.height * r)))
    display(pil)

    # 차이점 출력
    diffs = 0
    for a, b in zip(res_off, res_on):
        if a['helmet'] != b['helmet'] or a['vest'] != b['vest']:
            diffs += 1
    if diffs:
        print(f"  → {diffs}명에서 판단 결과 차이 발생")
    else:
        print(f"  → 판단 결과 동일")

## 8. 웹캠 실시간 테스트

In [ ]:
# 웹캠 초기화
from google.colab.output import eval_js

js = Javascript('''
    async function initCamera() {
        const video = document.createElement('video');
        video.width = 640; video.height = 480;
        const stream = await navigator.mediaDevices.getUserMedia(
            {video: {width: 640, height: 480}}
        );
        video.srcObject = stream;
        await video.play();
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        window._video = video;
        window._canvas = canvas;
        return [video.videoWidth, video.videoHeight];
    }
    async function captureFrame() {
        window._canvas.getContext('2d').drawImage(window._video, 0, 0);
        return window._canvas.toDataURL('image/jpeg', 0.8);
    }
    async function stopCamera() {
        if (window._video && window._video.srcObject)
            window._video.srcObject.getTracks().forEach(t => t.stop());
    }
''')
display(js)

dims = eval_js('initCamera()')
print(f"카메라 시작: {dims}")

In [ ]:
# ============================================
# 실시간 앙상블 추론
# 중지하려면: 런타임 > 실행 중단 (Ctrl+M+I)
# ============================================
from base64 import b64decode

N_FRAMES = 200           # 테스트할 프레임 수
USE_POSE = True          # Pose 사용 여부
POSE_EVERY_N = 3         # N프레임마다 Pose 실행 (1 = 매프레임)
DET_CONF = 0.4           # Detection confidence

frame_count = 0
violation_log = []       # 위반 기록

try:
    for i in range(N_FRAMES):
        # 프레임 캡처
        data_url = eval_js('captureFrame()')
        binary = b64decode(data_url.split(',')[1])
        arr = np.frombuffer(binary, dtype=np.uint8)
        frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if frame is None:
            continue

        frame_count += 1
        run_pose = USE_POSE and (frame_count % POSE_EVERY_N == 0)

        # 추론
        t0 = time.time()
        persons = run_pipeline(frame, det_model, pose_model,
                               det_conf=DET_CONF, use_pose=run_pose)
        dt = time.time() - t0
        fps = 1.0 / dt if dt > 0 else 0

        # 시각화
        vis, v_count = draw_frame(frame, persons)

        # HUD 정보
        pose_label = "ON" if run_pose else "SKIP"
        cv2.putText(vis, f"FPS:{fps:.1f} | Pose:{pose_label} | #{frame_count}",
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        if v_count > 0:
            cv2.putText(vis, f"VIOLATION: {v_count}",
                       (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            violation_log.append({
                'frame': frame_count,
                'count': v_count,
                'details': [(p['helmet'], p['vest']) for p in persons if not p['safe']]
            })

        # 표시
        vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
        pil = PIL.Image.fromarray(vis_rgb)
        buf = io.BytesIO()
        pil.save(buf, format='JPEG', quality=80)

        clear_output(wait=True)
        display(IPImage(data=buf.getvalue(), width=640))

        safe_count = sum(1 for p in persons if p['safe'])
        print(f"Frame {frame_count}/{N_FRAMES} | FPS: {fps:.1f} | "
              f"Persons: {len(persons)} | Safe: {safe_count} | Violations: {v_count}")

except KeyboardInterrupt:
    print("\n중단됨")
finally:
    eval_js('stopCamera()')
    print(f"\n카메라 종료")
    print(f"총 프레임: {frame_count}")
    print(f"위반 감지 프레임: {len(violation_log)}")

In [ ]:
# 위반 로그 요약
if violation_log:
    print(f"\n{'='*50}")
    print(f" 위반 로그 (총 {len(violation_log)} 프레임)")
    print(f"{'='*50}")
    for v in violation_log[-10:]:  # 마지막 10건
        print(f"  Frame #{v['frame']:4d} | 위반 {v['count']}건 | {v['details']}")
else:
    print("위반 없음")

## 9. 속도 벤치마크

In [ ]:
# Detection only vs Det+Pose 속도 비교
dummy = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

# Warm-up
for _ in range(5):
    run_pipeline(dummy, det_model, pose_model, use_pose=True)

N = 50

# (A) Detection only
t0 = time.time()
for _ in range(N):
    run_pipeline(dummy, det_model, pose_model, use_pose=False)
fps_det = N / (time.time() - t0)

# (B) Det + Pose 매프레임
t0 = time.time()
for _ in range(N):
    run_pipeline(dummy, det_model, pose_model, use_pose=True)
fps_both = N / (time.time() - t0)

# (C) Det + Pose 3프레임마다
t0 = time.time()
for i in range(N):
    run_pipeline(dummy, det_model, pose_model, use_pose=(i % 3 == 0))
fps_skip = N / (time.time() - t0)

print("\n" + "="*50)
print(" 속도 벤치마크 (Colab T4)")
print("="*50)
print(f" Detection only           : {fps_det:.1f} FPS")
print(f" Detection + Pose (every) : {fps_both:.1f} FPS  ({(1-fps_both/fps_det)*100:.0f}% drop)")
print(f" Detection + Pose (skip 3): {fps_skip:.1f} FPS  ({(1-fps_skip/fps_det)*100:.0f}% drop)")
print(f"\n → Hailo에서는 이보다 훨씬 낮아질 것으로 예상")
print(f" → skip 방식의 FPS 회복률이 핵심 발표 포인트")

## 10. 다음 단계

**로직 검증 완료 후 체크리스트:**

- [ ] 판단 임계값 조정 (현재: head 근처 15%, vest 영역 35%)
- [ ] 착용/휴대 구분이 잘 되는 케이스/안 되는 케이스 스크린샷 수집
- [ ] `POSE_EVERY_N` 최적값 결정 (FPS vs 정확도 트레이드오프)
- [ ] ByteTrack 트래커 추가 → 인물별 ID + 시간 기반 위반 로그
- [ ] 라즈베리파이 + Hailo 이식 (`run_pipeline` 내부만 HailoRT로 교체)